# Tópicos Especiais em Computação I
## Análise de Homicídios Mundiais

Equipe:

José Ericson Silveira Teófilo

Igor da Silva Pierre

Glória Maria Mesquita Furtado

Pedro Carolino Neto

Jeferson Rodrigo Silva de Mesquita

Antonio Lucas Damasceno Melo

Fonte dos dados:
- Escritório das Nações Unidas sobre Drogas e Crime (UNODC)
- Dataset: https://dataunodc.un.org/dp-intentional-homicide-victims
- Período: 2013 a 2022

Objetivo: Analisar o dataset com índices de homicídios em todo o mundo, utilizando estatística descritiva, bibliotecas de gráficos e Pandas para extrair informações importantes sobre esse fenômeno.

---
## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação das bibliotecas necessárias
!pip install pandas numpy matplotlib seaborn plotly openpyxl --quiet

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas importadas com sucesso!')
print(f'Pandas versão: {pd.__version__}')
print(f'NumPy versão: {np.__version__}')

Bibliotecas importadas com sucesso!
Pandas versão: 2.2.2
NumPy versão: 2.0.2


## 2. Carregamento dos Dados

O dataset possui duas abas:
- Aba 1 (`data_cts_intentional_homicide`): dados por país, com múltiplas dimensões e categorias
- Aba 2 (`data_cts_homicide_reg_estimates`): estimativas agregadas por região e sub-região

In [ ]:
# Fazendo o upload do arquivo no Google Colab
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
# Nome do arquivo enviado
ARQUIVO = '/content/sample_data/data_cts_intentional_homicide.xlsx'

# Carregando as duas abas do dataset
df_paises  = pd.read_excel(ARQUIVO, sheet_name='data_cts_intentional_homicide')
df_regioes = pd.read_excel(ARQUIVO, sheet_name='data_cts_homicide_reg_estimates')

print('Dados carregados com sucesso!')
print(f'Aba 1 (países):  {df_paises.shape[0]:,} linhas x {df_paises.shape[1]} colunas')
print(f'Aba 2 (regiões): {df_regioes.shape[0]:,} linhas x {df_regioes.shape[1]} colunas')

## 3. Exploração Inicial dos Dados

In [ ]:
# Visualizando as primeiras linhas da Aba 1 (países)
print('=== ABA 1 - Por País ===')
df_paises.head()

In [ ]:
# Visualizando as primeiras linhas da Aba 2(regiões)
print('=== ABA 2 - Por Região ===')
df_regioes.head()

In [ ]:
# Informações gerais da Aba 1
print('=== Informações da Aba 1 ===')
df_paises.info()

In [ ]:
# Informações gerais da Aba 2
print('=== Informações da Aba 2 ===')
df_regioes.info()

In [ ]:
# Estatísticas descritivas da Aba 1
print('=== Estatísticas Descritivas - Aba 1 ===')
df_paises.describe()

In [ ]:
# Estatísticas descritivas da Aba 2
print('=== Estatísticas Descritivas - Aba 2 ===')
df_regioes.describe()

In [ ]:
# Valores únicos das colunas categóricas da Aba 1
colunas_cat = ['Region', 'Subregion', 'Indicator', 'Dimension', 'Category', 'Sex', 'Unit of measurement']

for col in colunas_cat:
    print(f"\n{col}: {df_paises[col].unique()}")

In [ ]:
# Valores únicos das colunas categóricas da Aba 2
colunas_cat2 = ['region', 'subregion', 'indicator', 'series', 'sex', 'obs_status']

for col in colunas_cat2:
    print(f"\n{col}: {df_regioes[col].unique()}")

## 4. Verificação e Tratamento de Valores Nulos

In [ ]:
# Verificando valores nulos na Aba 1
print('=== Valores Nulos - Aba 1 ===')
total_nulos = df_paises.isnull().sum().sort_values(ascending=False)
porcent_nulos = ((df_paises.isnull().sum() / df_paises.shape[0]) * 100).sort_values(ascending=False)

resumo_nulos = pd.DataFrame({'Total': total_nulos, 'Porcentagem (%)': porcent_nulos})
print(resumo_nulos)

In [ ]:
# Verificando valores nulos na Aba 2
print('=== Valores Nulos - Aba 2 ===')
total_nulos2 = df_regioes.isnull().sum().sort_values(ascending=False)
porcent_nulos2 = ((df_regioes.isnull().sum() / df_regioes.shape[0]) * 100).sort_values(ascending=False)

resumo_nulos2 = pd.DataFrame({'Total': total_nulos2, 'Porcentagem (%)': porcent_nulos2})
print(resumo_nulos2)

In [ ]:
# Tratamento da Aba 2:
# Removendo linhas com obs_status = 'Omitted value for confidentiality reasons'
# pois esses registros não possuem valor numérico utilizável
df_regioes = df_regioes[df_regioes['obs_status'] == 'Estimated value'].copy()
df_regioes = df_regioes.reset_index(drop=True)

print(f'Aba 2 após remoção de valores omitidos: {df_regioes.shape[0]} linhas')
print(f'Valores nulos restantes: {df_regioes["value"].isnull().sum()}')

## 5. Criação dos Filtros Base

A Aba 1 contém múltiplas dimensões de análise (por mecanismo, por relacionamento, por cidadania etc.), o que faz com que um mesmo homicídio apareça em várias linhas. Para evitar **dupla contagem**, aplicamos o filtro base abaixo antes de qualquer análise por país.

In [ ]:
# Filtro base da Aba 1:
# Apenas homicídios intencionais (exclui presos, condenados, mortes na prisão)
# Apenas totais (evita dupla contagem por dimensão/categoria)
# Apenas contagens absolutas (não taxa por 100k)
df_base = df_paises[
    (df_paises['Indicator']            == 'Victims of intentional homicide') &
    (df_paises['Dimension']            == 'Total') &
    (df_paises['Category']             == 'Total') &
    (df_paises['Unit of measurement']  == 'Counts')
].copy()

df_base = df_base.reset_index(drop=True)

print(f'Linhas antes do filtro: {df_paises.shape[0]:,}')
print(f'Linhas após filtro base: {df_base.shape[0]:,}')
print(f'\nDistribuição por Sexo:')
print(df_base['Sex'].value_counts())

In [ ]:
# Sub-datasets por sexo (usados nas perguntas específicas)
df_total   = df_base[df_base['Sex'] == 'Total'].copy()   # todos os homicídios
df_female  = df_base[df_base['Sex'] == 'Female'].copy()  # homicídios de mulheres
df_male    = df_base[df_base['Sex'] == 'Male'].copy()    # homicídios de homens

print(f'Total (ambos os sexos): {df_total.shape[0]:,} linhas')
print(f'Somente mulheres:       {df_female.shape[0]:,} linhas')
print(f'Somente homens:         {df_male.shape[0]:,} linhas')

In [ ]:
# Sub-datasets da Aba 2 por série
df_reg_contagem = df_regioes[
    df_regioes['series'] == 'Number of victims of intentional homicide'
].copy()  # contagem absoluta por região

df_reg_taxa = df_regioes[
    df_regioes['series'] == 'Victims of intentional homicide per 100,000 population'
].copy()  # taxa por 100k por região

print(f'Regiões - contagem absoluta: {df_reg_contagem.shape[0]:,} linhas')
print(f'Regiões - taxa por 100k:     {df_reg_taxa.shape[0]:,} linhas')

## 6. Verificação Final dos Dados Prontos para Análise

In [ ]:
# Resumo dos DataFrames disponíveis para as análises
print('=' * 55)
print('DATAFRAMES DISPONÍVEIS PARA ANÁLISE')
print('=' * 55)
print(f'df_base       → todos os sexos, filtro base aplicado  : {df_base.shape}')
print(f'df_total      → Sex == Total                          : {df_total.shape}')
print(f'df_female     → Sex == Female                         : {df_female.shape}')
print(f'df_male       → Sex == Male                           : {df_male.shape}')
print(f'df_reg_contagem → regiões, contagem absoluta          : {df_reg_contagem.shape}')
print(f'df_reg_taxa     → regiões, taxa por 100k              : {df_reg_taxa.shape}')
print('=' * 55)
print('\nPronto para iniciar as análises!')

In [ ]:
# Visualizando amostra do df_total (base principal das análises por país)
print('Amostra do df_total:')
df_total.head(10)

## 7. Análise Exploratória — As 10 Perguntas

> **Nota:** Esta seção continua diretamente da Parte 1. Os DataFrames `df_total`, `df_female`, `df_reg_contagem` já devem estar carregados.

### Pergunta 1 — Quais países apresentam os 10 maiores índices de homicídios nos últimos 5 anos?

In [ ]:
# Filtrando os últimos 5 anos disponíveis no dataset (2018-2022)
# e somando o total de homicídios por país nesse período
top10_paises = (
    df_total[df_total['Year'] >= 2018]
    .groupby('Country')['VALUE']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10_paises.columns = ['País', 'Total de Homicídios']
top10_paises['Total de Homicídios'] = top10_paises['Total de Homicídios'].astype(int)
print(top10_paises.to_string(index=False))

In [ ]:
# Gráfico de barras horizontais para facilitar a leitura dos nomes dos países
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top10_paises,
    x='Total de Homicídios',
    y='País',
    palette='Reds_r'
)
plt.title('Top 10 Países com Mais Homicídios (2018–2022)', fontsize=14)
plt.xlabel('Total de Homicídios')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### Pergunta 2 — Quais países apresentam os 10 maiores índices de homicídios de mulheres em 2022?

In [ ]:
# Filtrando apenas o ano de 2022 no dataset de mulheres
top10_mulheres_2022 = (
    df_female[df_female['Year'] == 2022]
    .groupby('Country')['VALUE']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10_mulheres_2022.columns = ['País', 'Homicídios de Mulheres']
top10_mulheres_2022['Homicídios de Mulheres'] = top10_mulheres_2022['Homicídios de Mulheres'].round(0).astype(int)
print(top10_mulheres_2022.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top10_mulheres_2022,
    x='Homicídios de Mulheres',
    y='País',
    palette='Purples_r'
)
plt.title('Top 10 Países com Mais Homicídios de Mulheres em 2022', fontsize=14)
plt.xlabel('Número de Homicídios')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### Pergunta 3 — Quais as regiões com mais homicídios?

In [ ]:
# Usando a Aba 2 (estimativas regionais), excluindo a linha 'World'
# e somando o total histórico por região
regioes_total = (
    df_reg_contagem[
        (df_reg_contagem['sex'] == 'Total') &
        (df_reg_contagem['region'] != 'World')
    ]
    .groupby('region')['value']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
regioes_total.columns = ['Região', 'Total de Homicídios']
print(regioes_total.to_string(index=False))

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    data=regioes_total,
    x='Região',
    y='Total de Homicídios',
    palette='Oranges_r'
)
plt.title('Regiões com Mais Homicídios (2013–2022)', fontsize=14)
plt.xlabel('Região')
plt.ylabel('Total de Homicídios')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Pergunta 4 — Quais países com menor número de homicídios em cada sub-região?

In [ ]:
# Somando total histórico por sub-região e país
# e selecionando o país com menor valor em cada sub-região
total_por_subregiao_pais = (
    df_total
    .groupby(['Subregion', 'Country'])['VALUE']
    .sum()
    .reset_index()
)

menor_por_subregiao = (
    total_por_subregiao_pais
    .loc[total_por_subregiao_pais.groupby('Subregion')['VALUE'].idxmin()]
    .sort_values('Subregion')
    .reset_index(drop=True)
)
menor_por_subregiao.columns = ['Sub-região', 'País', 'Total de Homicídios']
menor_por_subregiao['Total de Homicídios'] = menor_por_subregiao['Total de Homicídios'].astype(int)
print(menor_por_subregiao.to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(
    data=menor_por_subregiao,
    x='Total de Homicídios',
    y='Sub-região',
    palette='Greens'
)
# Adicionando o nome do país em cada barra
for i, row in menor_por_subregiao.iterrows():
    plt.text(row['Total de Homicídios'] + 0.5, i, row['País'], va='center', fontsize=8)
plt.title('País com Menor Número de Homicídios por Sub-região', fontsize=14)
plt.xlabel('Total de Homicídios (2013–2022)')
plt.ylabel('Sub-região')
plt.tight_layout()
plt.show()

### Pergunta 5 — Quais países com menor número de morte de mulheres?

In [ ]:
# Somando o total histórico de homicídios de mulheres por país
# e exibindo os 10 países com menor número
menor_feminicidio = (
    df_female
    .groupby('Country')['VALUE']
    .sum()
    .sort_values(ascending=True)
    .head(10)
    .reset_index()
)
menor_feminicidio.columns = ['País', 'Total de Homicídios de Mulheres']
menor_feminicidio['Total de Homicídios de Mulheres'] = menor_feminicidio['Total de Homicídios de Mulheres'].round(0).astype(int)
print(menor_feminicidio.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=menor_feminicidio,
    x='Total de Homicídios de Mulheres',
    y='País',
    palette='Blues'
)
plt.title('Top 10 Países com Menor Número de Homicídios de Mulheres (2013–2022)', fontsize=13)
plt.xlabel('Total de Homicídios')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### Pergunta 6 — Quais as sub-regiões com maior número de homicídios?

In [ ]:
# Usando a Aba 2, excluindo a linha 'All' (que representa o total da região)
subregioes_total = (
    df_reg_contagem[
        (df_reg_contagem['sex'] == 'Total') &
        (df_reg_contagem['subregion'] != 'All')
    ]
    .groupby('subregion')['value']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
subregioes_total.columns = ['Sub-região', 'Total de Homicídios']
print(subregioes_total.to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(
    data=subregioes_total,
    x='Total de Homicídios',
    y='Sub-região',
    palette='Reds_r'
)
plt.title('Sub-regiões com Maior Número de Homicídios (2013–2022)', fontsize=14)
plt.xlabel('Total de Homicídios')
plt.ylabel('Sub-região')
plt.tight_layout()
plt.show()

### Pergunta 7 — Identifique o país com maior número de homicídios em cada continente em 2020

In [ ]:
# Filtrando o ano de 2020 e encontrando o país com maior VALUE em cada região
total_2020 = (
    df_total[df_total['Year'] == 2020]
    .groupby(['Region', 'Country'])['VALUE']
    .sum()
    .reset_index()
)

maior_por_continente_2020 = (
    total_2020
    .loc[total_2020.groupby('Region')['VALUE'].idxmax()]
    .sort_values('VALUE', ascending=False)
    .reset_index(drop=True)
)
maior_por_continente_2020.columns = ['Continente', 'País', 'Homicídios em 2020']
maior_por_continente_2020['Homicídios em 2020'] = maior_por_continente_2020['Homicídios em 2020'].astype(int)
print(maior_por_continente_2020.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=maior_por_continente_2020,
    x='Continente',
    y='Homicídios em 2020',
    palette='Oranges_r'
)
# Adicionando o nome do país em cima de cada barra
for i, row in maior_por_continente_2020.iterrows():
    plt.text(i, row['Homicídios em 2020'] + 300, row['País'], ha='center', fontsize=9)
plt.title('País com Maior Número de Homicídios por Continente em 2020', fontsize=13)
plt.xlabel('Continente')
plt.ylabel('Número de Homicídios')
plt.tight_layout()
plt.show()

### Pergunta 8 — Qual o país mais violento para as mulheres em 2021?

In [ ]:
# Filtrando o ano de 2021 no dataset de mulheres
# e ordenando pelo número de homicídios
violento_mulheres_2021 = (
    df_female[df_female['Year'] == 2021]
    .groupby('Country')['VALUE']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
violento_mulheres_2021.columns = ['País', 'Homicídios de Mulheres em 2021']
violento_mulheres_2021['Homicídios de Mulheres em 2021'] = violento_mulheres_2021['Homicídios de Mulheres em 2021'].round(0).astype(int)

print(f"País mais violento para mulheres em 2021: {violento_mulheres_2021.iloc[0]['País']}")
print(f"Total de homicídios: {violento_mulheres_2021.iloc[0]['Homicídios de Mulheres em 2021']:,}")
print()
print(violento_mulheres_2021.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=violento_mulheres_2021,
    x='Homicídios de Mulheres em 2021',
    y='País',
    palette='Purples_r'
)
plt.title('Top 10 Países Mais Violentos para Mulheres em 2021', fontsize=14)
plt.xlabel('Número de Homicídios')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### Pergunta 9 — Qual o país com maior valor do indicador 'Victims of intentional homicide'?

In [ ]:
# Somando o total histórico(2013 a 2022) do indicador por país
maior_indicador = (
    df_total
    .groupby('Country')['VALUE']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
maior_indicador.columns = ['País', 'Total Histórico de Homicídios']
maior_indicador['Total Histórico de Homicídios'] = maior_indicador['Total Histórico de Homicídios'].astype(int)

print(f"País com maior valor histórico do indicador: {maior_indicador.iloc[0]['País']}")
print(f"Total: {maior_indicador.iloc[0]['Total Histórico de Homicídios']:,}")
print()
print(maior_indicador.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=maior_indicador,
    x='Total Histórico de Homicídios',
    y='País',
    palette='Reds_r'
)
plt.title("Top 10 Países — Indicator 'Victims of Intentional Homicide' (2013–2022)", fontsize=13)
plt.xlabel('Total de Homicídios')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### Pergunta 10 — Qual a média de homicídios no Brasil nos últimos 10 anos?

In [ ]:
# filtrando apenas o Brasil e calculando a média anual de homicidios
brasil = (
    df_total[df_total['Country'] == 'Brazil']
    [['Year', 'VALUE']]
    .sort_values('Year')
    .reset_index(drop=True)
)
brasil.columns = ['Ano', 'Homicídios']
brasil['Homicídios'] = brasil['Homicídios'].astype(int)

media_brasil = brasil['Homicídios'].mean()

print(brasil.to_string(index=False))
print(f'\nMédia anual de homicídios no Brasil (2013–2022): {media_brasil:,.2f}')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(brasil['Ano'], brasil['Homicídios'], marker='o', color='steelblue', linewidth=2, label='Homicídios por ano')
plt.axhline(y=media_brasil, color='red', linestyle='--', linewidth=1.5, label=f'Média: {media_brasil:,.0f}')
plt.title('Homicídios no Brasil por Ano (2013–2022)', fontsize=14)
plt.xlabel('Ano')
plt.ylabel('Número de Homicídios')
plt.xticks(brasil['Ano'])
plt.legend()
plt.tight_layout()
plt.show()

## 8. Análise de Regressão — Previsão de Homicídios (2023–2026)

Objetivo: Utilizar Regressão Linear para prever a taxa de homicídios nos anos de 2023, 2024, 2025 e 2026, com base nos dados históricos de 2013 a 2022.

Estratégia: A regressão é aplicada individualmente por país, usando o `Year` como variável preditora (X) e o número de homicídios (`VALUE`) como variável alvo (y). Ao final, é possível selecionar qualquer país e visualizar as previsões.

> Nota: Esta seção continua diretamente das Partes 1 e 2. O DataFrame `df_total` já deve estar carregado.

### 8.1 Importações adicionais para Machine Learning

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, max_error

print('Bibliotecas de Machine Learning importadas com sucesso!')

### 8.2 Função de Regressão por País

In [ ]:
def regressao_por_pais(pais, df=df_total, anos_previsao=[2023, 2024, 2025, 2026]):
    '''
    Aplica regressão linear para um país específico e retorna
    o modelo treinado, as métricas e as previsões futuras.

    :param pais: nome do país (string)
    :param df: DataFrame filtrado com Sex==Total
    :param anos_previsao: lista de anos a prever
    :return: dicionário com modelo, métricas e previsões
    '''
    # Filtrando e ordenando os dados do país
    dados = df[df['Country'] == pais][['Year', 'VALUE']].sort_values('Year').dropna()

    if len(dados) < 5:
        print(f'Dados insuficientes para {pais} ({len(dados)} anos disponíveis).')
        return None

    X = dados[['Year']].values
    y = dados['VALUE'].values

    # Divisão treino/teste: 70% treino, 30% teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

    # Treinando o modelo
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    # Previsões no conjunto de teste
    previsoes_test = modelo.predict(X_test)

    # métricas de avaliação
    metricas = {
        'R²':   round(modelo.score(X, y), 4),
        'MAE':  round(mean_absolute_error(y_test, previsoes_test), 2),
        'MSE':  round(mean_squared_error(y_test, previsoes_test), 2),
        'RMSE': round(mean_squared_error(y_test, previsoes_test) ** 0.5, 2),
        'Erro Máximo': round(max_error(y_test, previsoes_test), 2)
    }

    # Previso~es para os anos futuros
    X_futuro = np.array([[a] for a in anos_previsao])
    valores_futuros = modelo.predict(X_futuro)

    previsoes = dict(zip(anos_previsao, [max(0, v) for v in valores_futuros]))

    return {
        'modelo':    modelo,
        'dados':     dados,
        'metricas':  metricas,
        'previsoes': previsoes
    }

print('Função regressao_por_pais() definida com sucesso!')

### 8.3 Regressão — Brasil

In [ ]:
# Aplicando a regressão para o Brasil
resultado_brasil = regressao_por_pais('Brazil')

print('=== BRASIL ===')
print('\nMétricas de avaliação:')
for metrica, valor in resultado_brasil['metricas'].items():
    print(f'  {metrica}: {valor:,.2f}')

print('\nPrevisões de homicídios:')
for ano, valor in resultado_brasil['previsoes'].items():
    print(f'  {ano}: {valor:,.0f}')

print(f"\nCoeficiente (inclinação): {resultado_brasil['modelo'].coef_[0]:,.2f}")
print(f"Intercepto: {resultado_brasil['modelo'].intercept_:,.2f}")

In [ ]:
# Visualizando os dados históricos e as previsões do Brasil.
dados_br    = resultado_brasil['dados']
previsoes_br = resultado_brasil['previsoes']
modelo_br   = resultado_brasil['modelo']

# Linha de regressão sobre os dados históricos
anos_hist = dados_br['Year'].values
y_hist    = dados_br['VALUE'].values
y_linha   = modelo_br.predict(dados_br[['Year']].values)

# Anos e valores futuros
anos_fut  = list(previsoes_br.keys())
vals_fut  = list(previsoes_br.values())

plt.figure(figsize=(12, 6))
plt.plot(anos_hist, y_hist, 'o-', color='steelblue', linewidth=2, label='Dados reais (2013–2022)')
plt.plot(anos_hist, y_linha, '--', color='gray', linewidth=1.5, label='Linha de regressão')
plt.plot(anos_fut, vals_fut, 's--', color='red', linewidth=2, markersize=8, label='Previsão (2023–2026)')

# Anotando os valores previstos
for ano, val in zip(anos_fut, vals_fut):
    plt.annotate(f'{val:,.0f}', (ano, val), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

plt.axvline(x=2022.5, color='black', linestyle=':', linewidth=1, label='Início da previsão')
plt.title('Brasil — Homicídios Reais e Previsão (Regressão Linear)', fontsize=14)
plt.xlabel('Ano')
plt.ylabel('Número de Homicídios')
plt.xticks(list(anos_hist) + anos_fut, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### 8.4 Regressão — Países do Top 10 (Comparativo)

In [ ]:
# Aplicando a regressão para os 10 países com mais homicídios
# (resultado da primeira pergunta)
paises_top10 = top10_paises['País'].tolist()

resultados = {}
tabela_previsoes = []

for pais in paises_top10:
    res = regressao_por_pais(pais)
    if res:
        resultados[pais] = res
        linha = {'País': pais, 'R²': res['metricas']['R²']}
        linha.update(res['previsoes'])
        tabela_previsoes.append(linha)

# Tabela comparativa
df_previsoes = pd.DataFrame(tabela_previsoes)
df_previsoes[[2023, 2024, 2025, 2026]] = df_previsoes[[2023, 2024, 2025, 2026]].round(0).astype(int)
print('Previsões de Homicídios para os Top 10 Países (2023–2026):')
print(df_previsoes.to_string(index=False))

In [ ]:
# gráfico comparativo: previsão 2026 para os top 10 países
df_prev_2026 = df_previsoes[['País', 2026]].sort_values(2026, ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_prev_2026, x=2026, y='País', palette='Reds_r')
plt.title('Previsão de Homicídios em 2026 — Top 10 Países', fontsize=14)
plt.xlabel('Previsão de Homicídios')
plt.ylabel('País')
plt.tight_layout()
plt.show()

### 8.5 Regressão — Tendência Mundial (soma por ano)

In [ ]:
# Somando os homicidios de todos os países por ano
# para obter a tendencia global
mundial = (
    df_total
    .groupby('Year')['VALUE']
    .sum()
    .reset_index()
    .sort_values('Year')
)

X_mun = mundial[['Year']].values
y_mun = mundial['VALUE'].values

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_mun, y_mun, test_size=0.3, random_state=0)

modelo_mundial = LinearRegression()
modelo_mundial.fit(X_train_m, y_train_m)
prev_test_m = modelo_mundial.predict(X_test_m)

print('=== TENDÊNCIA MUNDIAL ===')
print(f'  R²:   {modelo_mundial.score(X_mun, y_mun):.4f}')
print(f'  MAE:  {mean_absolute_error(y_test_m, prev_test_m):,.2f}')
print(f'  RMSE: {mean_squared_error(y_test_m, prev_test_m)**0.5:,.2f}')

anos_futuros = np.array([[2023],[2024],[2025],[2026]])
prev_mundial = modelo_mundial.predict(anos_futuros)

print('\nPrevisões mundiais:')
for ano, val in zip([2023, 2024, 2025, 2026], prev_mundial):
    print(f'  {ano}: {val:,.0f}')

In [ ]:
# Gráfico da tendencia mundial com previsão
y_linha_mun = modelo_mundial.predict(X_mun)

plt.figure(figsize=(12, 6))
plt.plot(mundial['Year'], mundial['VALUE'], 'o-', color='steelblue', linewidth=2, label='Total mundial real')
plt.plot(mundial['Year'], y_linha_mun, '--', color='gray', linewidth=1.5, label='Linha de regressão')
plt.plot([2023, 2024, 2025, 2026], prev_mundial, 's--', color='red', linewidth=2, markersize=8, label='Previsão (2023–2026)')

for ano, val in zip([2023, 2024, 2025, 2026], prev_mundial):
    plt.annotate(f'{val:,.0f}', (ano, val), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

plt.axvline(x=2022.5, color='black', linestyle=':', linewidth=1, label='Início da previsão')
plt.title('Tendência Mundial de Homicídios e Previsão (Regressão Linear)', fontsize=14)
plt.xlabel('Ano')
plt.ylabel('Total de Homicídios')
plt.xticks(list(mundial['Year']) + [2023, 2024, 2025, 2026], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### 8.6 Previsão para Qualquer País

In [ ]:
# altere o valor da variável abaixo para o país desejado
PAIS_ESCOLHIDO = 'Brazil'  # <- alterar aqui

res = regressao_por_pais(PAIS_ESCOLHIDO)

if res:
    dados_p    = res['dados']
    previsoes_p = res['previsoes']
    modelo_p   = res['modelo']

    print(f'=== {PAIS_ESCOLHIDO.upper()} ===')
    print('\nMétricas:')
    for k, v in res['metricas'].items():
        print(f'  {k}: {v:,.2f}')

    print('\nPrevisões:')
    for ano, val in previsoes_p.items():
        print(f'  {ano}: {val:,.0f}')

    # Gráfico
    anos_hist_p = dados_p['Year'].values
    y_hist_p    = dados_p['VALUE'].values
    y_linha_p   = modelo_p.predict(dados_p[['Year']].values)
    anos_fut_p  = list(previsoes_p.keys())
    vals_fut_p  = list(previsoes_p.values())

    plt.figure(figsize=(12, 6))
    plt.plot(anos_hist_p, y_hist_p, 'o-', color='steelblue', linewidth=2, label='Dados reais')
    plt.plot(anos_hist_p, y_linha_p, '--', color='gray', linewidth=1.5, label='Linha de regressão')
    plt.plot(anos_fut_p, vals_fut_p, 's--', color='red', linewidth=2, markersize=8, label='Previsão (2023–2026)')

    for ano, val in zip(anos_fut_p, vals_fut_p):
        plt.annotate(f'{val:,.0f}', (ano, val), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

    plt.axvline(x=2022.5, color='black', linestyle=':', linewidth=1, label='Início da previsão')
    plt.title(f'{PAIS_ESCOLHIDO} — Homicídios Reais e Previsão (Regressão Linear)', fontsize=14)
    plt.xlabel('Ano')
    plt.ylabel('Número de Homicídios')
    plt.xticks(list(anos_hist_p) + anos_fut_p, rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()